# **Project 2: Joint Detection of AI-Generated Images and Post-Processing Alterations in Real-World Scenarios**

#**Imports**


In [ ]:
import os
import json
import random
import zipfile
import tarfile
from pathlib import Path
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision
import torchvision.transforms as transforms
from torchvision import models

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

from google.colab import drive

!pip install -q tqdm requests
import requests
from tqdm.auto import tqdm

import copy
import csv
import time

!pip install -q seaborn
import seaborn as sns

import io
from PIL import ImageFilter


#**Globals**

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
  print("GPU:", torch.cuda.get_device_name(0))

ROOT_DIR = Path("/content")
DATA_DIR = ROOT_DIR / "RRDataset"
DOWNLOAD_DIR = ROOT_DIR / "downloads"
EXTRACT_DIR = ROOT_DIR / "RRDataset_extracted"
CHECKPOINT_DIR = ROOT_DIR / "checkpoints"

DOWNLOAD_DIR.mkdir(exist_ok=True)
EXTRACT_DIR.mkdir(exist_ok=True)
CHECKPOINT_DIR.mkdir(exist_ok=True)

IMG_SIZE = 224
BATCH_SIZE = 16
NUM_EPOCHS = 10
LEARNING_RATE = 1e-4

RF_LABELS = {"real": 0, "fake": 1}
TRANS_LABELS = {
    "original": 0,
    "internet_transmitted": 1,
    "re_digitized": 2
}

NUM_WORKERS = 0
PATIENCE = 3
MIN_DELTA = 1e-4
BEST_MODEL_PATH = CHECKPOINT_DIR / "best_multitask_model.pt"
METRICS_PATH = CHECKPOINT_DIR / "multitask_metrics.json"
HISTORY_CSV_PATH = CHECKPOINT_DIR / "multitask_history.csv"

# **Utils**

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def show_image(image_path):
    img = Image.open(image_path).convert("RGB")
    plt.figure(figsize=(4,4))
    plt.imshow(img)
    plt.axis("off")
    plt.show()

def list_files_recursively(root_path, max_items=100):
    count = 0
    for root, dirs, files in os.walk(root_path):
        level = root.replace(str(root_path), "").count(os.sep)
        indent = " " * 2 * level
        print(f"{indent}{os.path.basename(root)}/")
        subindent = " " * 2 * (level + 1)
        for f in files:
            print(f"{subindent}{f}")
            count += 1
            if count >= max_items:
                print("\n[Output truncated]")
                return

In [ ]:
def download_with_progress(url, output_path, chunk_size=1024*1024):
    response = requests.get(url, stream=True)
    response.raise_for_status()

    total_size = int(response.headers.get("content-length", 0))

    with open(output_path, "wb") as f, tqdm(
        desc=output_path.name,
        total=total_size,
        unit="B",
        unit_scale=True,
        unit_divisor=1024,
    ) as bar:
        for chunk in response.iter_content(chunk_size=chunk_size):
            if chunk:
                f.write(chunk)
                bar.update(len(chunk))

    print(f"Download completato: {output_path}")

In [ ]:
# Questa funzione controlla se un'immagine è leggibile prima di inserirla nel dataset.
# In questo modo scartiamo i file corrotti una sola volta e non durante ogni epoca.
def is_valid_image(image_path):
    try:
        with Image.open(image_path) as img:
            img.verify()
        with Image.open(image_path) as img:
            img.convert("RGB")
        return True
    except Exception:
        return False

In [ ]:
def plot_confusion_matrix(cm, class_names, title="Confusion Matrix"):
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(title)
    plt.tight_layout()
    plt.show()

In [ ]:
# Salviamo la history del training in CSV così possiamo analizzarla anche fuori dal notebook.
# Questo è utile per creare grafici e tabelle per il report o per le slide.
def save_history_to_csv(history, csv_path):
    keys = list(history.keys())
    rows = zip(*[history[k] for k in keys])

    with open(csv_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(keys)
        writer.writerows(rows)


def save_metrics_to_json(metrics_dict, json_path):
    with open(json_path, "w") as f:
        json.dump(metrics_dict, f, indent=4)

#**Data**

## **2. Data Preparation e Simulazione di Robustezza**

### **Scelte Implementative sulla Selezione del Subset**
Poiché l'hardware di Colab (Tesla T4) pone vincoli di memoria e tempo, utilizziamo il pacchetto `RRDataset_original_train_val.tar.gz` (2.16 GB), che contiene 3000 immagini native ad alta risoluzione suddivise equamente in `real` e `ai`.

Per estendere il dataset al secondo task multi-classe (le 3 trasformazioni) senza saturare lo storage locale, implementiamo una pipeline di **Data Augmentation Dinamica on-the-fly** all'interno del modulo `Dataset` di PyTorch. Moltiplichiamo virtualmente il dataset per 3:
1. **Classe 0 (Original):** Immagine nativa pulita.
2. **Classe 1 (Internet-Transmitted):** Simulazione dei canali social (ridimensionamento bilineare dimezzato e forte compressione JPEG con fattore di qualità Q=30).
3. **Classe 2 (Re-Digitized):** Simulazione di ridigitalizzazione tramite applicazione di un filtro di sfocatura Gaussiano (Gaussian Blur, r=1.5) per emulare la degradazione della lente e del sensore d'acquisizione.

Questo approccio garantisce un **perfetto bilanciamento delle classi** per entrambi i task, eliminando qualsiasi bias statistico iniziale.

## Data Augmentation

La **Data Augmentation Dinamica on-the-fly** è già implementata nella cella successiva (`upyw6coL9H6e`) all'interno della classe `MultiTaskRRDataset`. Questo approccio aumenta il numero effettivo di immagini disponibili per l'addestramento senza la necessità di pre-generare e salvare grandi quantità di dati su disco. Ecco come funziona:

Per ogni immagine originale presente nel dataset, vengono generate e utilizzate 3 versioni distinte durante il training:
1.  **Originale:** L'immagine così come è stata acquisita, senza alterazioni.
2.  **Internet-Transmitted:** Una simulazione di come l'immagine potrebbe apparire dopo essere stata trasmessa su internet, tipicamente comportando ridimensionamento (qui, a metà risoluzione e poi riportata alla dimensione originale con ricampionamento bilineare) e forte compressione JPEG (con qualità Q=30).
3.  **Re-Digitized:** Una simulazione di una ri-digitalizzazione dell'immagine, ottenuta applicando un filtro di sfocatura gaussiana (`GaussianBlur` con raggio 1.5) per emulare la degradazione del sensore o della lente.

Oltre a queste 3 versioni generate, ogni singola immagine (originale, internet-transmitted, re-digitized) viene ulteriormente modificata da una serie di **trasformazioni casuali** definite in `train_transform`. Queste includono:
*   `RandomHorizontalFlip()`: Capovolgimento orizzontale casuale.
*   `RandomRotation(15)`: Rotazione casuale fino a 15 gradi.
*   `ColorJitter(...)`: Variazioni casuali di luminosità, contrasto, saturazione e tonalità.
*   `RandomResizedCrop(...)`: Ritaglio e ridimensionamento casuale per introdurre variazioni di scala e prospettiva.

Questo processo di aumento dinamico triplica il numero di campioni base e poi lo arricchisce ulteriormente con le trasformazioni casuali, rendendo il modello più robusto e generalizzabile. Puoi verificare il numero di campioni aumentati nell'output della cella seguente.

### Download and Extract RRDataset
This cells downloads the RRDataset from Zenodo and extracts it into the local environment.

##Real file recovery from Zenodo

In [ ]:
import urllib.request

ZENODO_RECORD = "14963880"
ZENODO_API = f"https://zenodo.org/api/records/{ZENODO_RECORD}"

with urllib.request.urlopen(ZENODO_API) as response:
    record_meta = json.loads(response.read().decode())

files_in_record = record_meta.get("files", [])

print("File trovati nel record Zenodo:")
for i, f in enumerate(files_in_record):
    print(f"{i}: {f['key']}  |  {round(f.get('size', 0)/1e9, 2)} GB")

print("\nDettaglio completo dei file:\n")
for f in files_in_record:
    print("Nome file:", f["key"])
    print("Dimensione (GB):", round(f.get("size", 0)/1e9, 2))
    print("Link diretto:", f["links"]["self"])
    print("-" * 60)

##Download Dataset

In [ ]:
target_file = files_in_record[1]   # Changed to download the larger test dataset
target_url = target_file["links"]["self"]
target_name = target_file["key"]

dest_path = DOWNLOAD_DIR / target_name

if not dest_path.exists():
    print("Scaricamento:", target_name)
    download_with_progress(target_url, dest_path)
else:
    print("File già presente:", dest_path)

##Archive extraction

In [ ]:
import hashlib

def calculate_md5(file_path):
    hash_md5 = hashlib.md5()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            hash_md5.update(chunk)
    return hash_md5.hexdigest()

archive_path = dest_path

if not archive_path.exists():
    print(f"Errore: Il file archivio non esiste: {archive_path}. Si prega di eseguire la cella di download (precedente a questa).")
elif str(archive_path).endswith(".zip"):
    try:
        with zipfile.ZipFile(archive_path, "r") as zf:
            zf.extractall(EXTRACT_DIR)
        print("Archivio ZIP estratto con successo in:", EXTRACT_DIR)
    except Exception as e:
        print(f"Errore durante l'estrazione del file ZIP: {e}")
        print(f"Si prega di verificare l'integrità del file o di eliminarlo manualmente: {archive_path} e riprovare a scaricarlo.")
elif str(archive_path).endswith(".tar.gz"):
    print(f"Verifica integrità per {archive_path}...")
    # The 'target_file' variable should be available from a previous cell's execution
    expected_checksum_full = target_file["checksum"] # e.g., 'md5:2f4498c3690d8f4c7a30d2e41dd34500'
    expected_md5 = expected_checksum_full.split(":")[1]

    calculated_md5 = calculate_md5(archive_path)

    if calculated_md5 == expected_md5:
        print("Checksum MD5 verificato. Il file è integro. Procedo con l'estrazione.")
        try:
            with tarfile.open(archive_path, "r:gz") as tf:
                tf.extractall(EXTRACT_DIR)
            print("Archivio TAR.GZ estratto con successo in:", EXTRACT_DIR)
        except EOFError:
            print(f"EOFError durante l'estrazione di {archive_path}. Il file potrebbe essere parzialmente corrotto nonostante il checksum o ci sono problemi con il formato dell'archivio.")
            print(f"Si prega di eliminare manualmente il file corrotto: {archive_path} e rieseguire la cella di download per un nuovo tentativo.")
        except Exception as e:
            print(f"Errore durante l'estrazione del file TAR.GZ: {e}")
            print(f"Si prega di verificare l'integrità del file o di eliminarlo manualmente: {archive_path} e riprovare a scaricarlo.")
    else:
        print(f"Errore di integrità: Checksum MD5 non corrispondente per {archive_path}.")
        print(f"Calcolato: {calculated_md5}, Atteso: {expected_md5}")
        print(f"Il file è corrotto. Si prega di eliminarlo manualmente: {archive_path} e rieseguire la cella di download per scaricarlo nuovamente.")
        print("Skipping extraction due to file corruption.")
else:
    print(f"Tipo di archivio non supportato: {archive_path}. Supportati .zip e .tar.gz")

print("Prime cartelle trovate:")
# Ensure EXTRACT_DIR exists even if extraction failed, to prevent error on rglob
EXTRACT_DIR.mkdir(exist_ok=True)
for p in sorted(EXTRACT_DIR.rglob("*")):
    if p.is_dir():
        print(p)

##Dataset structure inspection

In [ ]:
#print("Struttura del dataset estratto:\n")
#list_files_recursively(EXTRACT_DIR, max_items=200)

##Image count

In [ ]:
IMG_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

all_images = []
for p in EXTRACT_DIR.rglob("*"):
    if p.suffix.lower() in IMG_EXTENSIONS:
        all_images.append(p)

print("Numero totale immagini trovate:", len(all_images))

#for p in all_images[:20]:
#    print(p)

Implementazione della Pipline Dati Multi-Task

In [ ]:
class MultiTaskRRDataset(Dataset):
    def __init__(self, root_dir, split="train", transform=None, validate_files=True):

        # Qui costruiamo la lista dei campioni validi del dataset.
        # Se validate_files=True, i file corrotti vengono esclusi subito.
        # Modified to use the larger RRDataset_final
        self.split_dir = Path(root_dir) / "RRDataset_final" / split
        if not self.split_dir.exists() and split == "train": # Handle 'train' for test dataset
            self.split_dir = Path(root_dir) / "RRDataset_final" / "original" # Assuming original is where the train/val split is within the larger dataset

        self.transform = transform
        self.samples = []

        # Collect samples for original/ai and original/real for main RF task
        for label_name, label_idx in RF_LABELS.items():
            folder = Path(root_dir) / "RRDataset_final" / "original" / ("ai" if label_name == "fake" else "real")
            if folder.exists():
                img_list = sorted(folder.glob("*"))
                for img_p in img_list:
                    if img_p.suffix.lower() in IMG_EXTENSIONS:
                        if (not validate_files) or is_valid_image(img_p):
                            self.samples.append((img_p, label_idx))
                        else:
                            print(f"File corrotto scartato: {img_p}")

        print(f"[{split}] Campioni validi trovati: {len(self.samples)}")

    def __len__(self):
        # Ogni immagine originale viene usata in 3 versioni:
        # originale, internet-transmitted simulata e re-digitized simulata.
        return len(self.samples) * 3

    def __getitem__(self, idx):
        # base_idx seleziona l'immagine di partenza.
        # trans_idx decide quale trasformazione applicare tra le 3 classi del task secondario.
        base_len = len(self.samples)
        base_idx = idx % base_len
        trans_idx = idx // base_len

        img_path, rf_label = self.samples[base_idx]
        img = Image.open(img_path).convert("RGB")

        # Reverting to the original on-the-fly augmentation logic as described in the notebook.
        # The previous attempt to load from 'transfer'/'redigital' folders caused FileNotFoundError
        # because not all images have direct counterparts in those physical subfolders.
        if trans_idx == 1: # Internet-Transmitted (simulation)
            img = img.resize((IMG_SIZE // 2, IMG_SIZE // 2), Image.Resampling.BILINEAR)
            img = img.resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR)
            buffer = io.BytesIO()
            img.save(buffer, format="JPEG", quality=30)
            buffer.seek(0)
            img = Image.open(buffer).convert("RGB")

        elif trans_idx == 2: # Re-digitized (simulation)
            img = img.resize((IMG_SIZE, IMG_SIZE))
            img = img.filter(ImageFilter.GaussianBlur(radius=1.5))

        if self.transform:
            img = self.transform(img)

        return (
            img,
            torch.tensor(rf_label, dtype=torch.long),
            torch.tensor(trans_idx, dtype=torch.long)
        )

# Definizione dei Trasformatori PyTorch
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15), # New: Randomly rotate by up to 15 degrees
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1), # New: Randomly change colors
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)), # New: Randomly crop and resize
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Initialize with the correct root directory for the extracted large dataset
train_dataset = MultiTaskRRDataset(EXTRACT_DIR, split="train", transform=train_transform)
val_dataset = MultiTaskRRDataset(EXTRACT_DIR, split="val", transform=val_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print(f"Campioni totali di Addestramento: {len(train_dataset)}")
print(f"Campioni totali di Validazione: {len(val_dataset)}")

#**Network**

## **3. Modello Unificato Multi-Task (Shared Backbone)**

### **Scelte Architetturali**
Per mappare simultaneamente feature eterogenee (il riconoscimento di pattern generativi richiede l'analisi di dettagli microscopici ad alta frequenza, mentre il tipo di alterazione richiede l'analisi globale della tessitura e della sfocatura), utilizziamo una **ResNet50** preaddestrata su ImageNet come **Backbone Condiviso**.

Rimuoviamo lo strato di classificazione nativo (`fc = nn.Identity()`) e colleghiamo due teste lineari parallele e indipendenti allo spazio delle feature estratte (vettore a 2048 dimensioni):
* **Head Real/Fake (`head_rf`):** Uno strato lineare fully-connected che mappa le feature in 2 classi (reale vs IA).
* **Head Transformation (`head_trans`):** Uno strato lineare parallelo che mappa le medesime feature in 3 classi (tipo di post-processing).

In [ ]:
class MultiTaskModel(nn.Module):
    def __init__(self, backbone_name="resnet50", pretrained=True):
        super(MultiTaskModel, self).__init__()

        if backbone_name == "resnet50":
            weights = models.ResNet50_Weights.DEFAULT if pretrained else None
            self.backbone = models.resnet50(weights=weights)
            num_features = self.backbone.fc.in_features
            self.backbone.fc = nn.Identity()
        else:
            raise ValueError("Backbone non implementato.")

        # Head 1: Classificazione Binaria Real/Fake
        self.head_rf = nn.Linear(num_features, 2)

        # Head 2: Classificazione Multi-classe delle Alterazioni
        self.head_trans = nn.Linear(num_features, 3)

    def forward(self, x):
        # Condivisione delle feature di basso e alto livello
        features = self.backbone(x)

        # Inferenza parallela sui due task
        out_rf = self.head_rf(features)
        out_trans = self.head_trans(features)
        return out_rf, out_trans

model = MultiTaskModel(backbone_name="resnet50", pretrained=True).to(DEVICE)
print(f"Modello inizializzato con successo. Parametri totali: {count_parameters(model):,}")

In [ ]:
pip install -q torchsummary

In [ ]:
from torchsummary import summary

# Esempio di input per la rete (simulando la dimensione delle immagini)
summary(model, input_size=(3, IMG_SIZE, IMG_SIZE))

# **Train**

## **4. Funzione di Loss Congiunta e Addestramento**

La loss di ottimizzazione globale del modello viene definita come la combinazione lineare pesata delle singole Cross-Entropy Loss dei due task:

$$L_{total} = \alpha \cdot L_{RF} + \beta \cdot L_{Trans}$$

In questa configurazione iniziale di baseline impostiamo $\alpha = 1.0$ e $\beta = 1.0$, assegnando pari dignità algoritmica a entrambi gli obiettivi. L'ottimizzatore scelto è **AdamW** (con `learning_rate = 1e-4` e un leggero `weight_decay`), ideale per prevenire l'overfitting delle teste lineari durante il fine-tuning del backbone.

In [ ]:
criterion_rf = nn.CrossEntropyLoss()
criterion_trans = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

ALPHA = 1.0  # Peso associato al Task Real/Fake
BETA = 1.0   # Peso associato al Task Post-Processing

def train_epoch(model, dataloader, optimizer, device):
    model.train()
    running_loss = 0.0
    correct_rf, correct_trans = 0, 0
    total = 0

    for imgs, rf_labels, trans_labels in tqdm(dataloader, desc="Training"):
        imgs, rf_labels, trans_labels = imgs.to(device), rf_labels.to(device), trans_labels.to(device)

        optimizer.zero_grad()
        out_rf, out_trans = model(imgs)

        loss_rf = criterion_rf(out_rf, rf_labels)
        loss_trans = criterion_trans(out_trans, trans_labels)

        # Loss Multi-Task Combinata
        total_loss = (ALPHA * loss_rf) + (BETA * loss_trans)
        total_loss.backward()
        optimizer.step()

        running_loss += total_loss.item() * imgs.size(0)

        _, pred_rf = torch.max(out_rf, 1)
        _, pred_trans = torch.max(out_trans, 1)

        correct_rf += (pred_rf == rf_labels).sum().item()
        correct_trans += (pred_trans == trans_labels).sum().item()
        total += imgs.size(0)

    return running_loss / total, correct_rf / total, correct_trans / total

def validate(model, dataloader, device):
    model.eval()
    running_loss = 0.0
    all_rf_preds, all_rf_labels = [], []
    all_trans_preds, all_trans_labels = [], []

    with torch.no_grad():
        for imgs, rf_labels, trans_labels in dataloader:
            imgs, rf_labels, trans_labels = imgs.to(device), rf_labels.to(device), trans_labels.to(device)

            out_rf, out_trans = model(imgs)
            loss_rf = criterion_rf(out_rf, rf_labels)
            loss_trans = criterion_trans(out_trans, trans_labels)
            total_loss = (ALPHA * loss_rf) + (BETA * loss_trans)

            running_loss += total_loss.item() * imgs.size(0)

            _, pred_rf = torch.max(out_rf, 1)
            _, pred_trans = torch.max(out_trans, 1)

            all_rf_preds.extend(pred_rf.cpu().numpy())
            all_rf_labels.extend(rf_labels.cpu().numpy())
            all_trans_preds.extend(pred_trans.cpu().numpy())
            all_trans_labels.extend(trans_labels.cpu().numpy())

    total = len(dataloader.dataset)
    acc_rf = accuracy_score(all_rf_labels, all_rf_preds)
    acc_trans = accuracy_score(all_trans_labels, all_trans_preds)

    return running_loss / total, acc_rf, acc_trans, all_rf_preds, all_rf_labels, all_trans_preds, all_trans_labels

Esecuzione del Training Loop


In [ ]:

history = {
    "train_loss": [],
    "train_acc_rf": [],
    "train_acc_trans": [],
    "val_loss": [],
    "val_acc_rf": [],
    "val_acc_trans": []
}

# Teniamo traccia del miglior modello in base alla validation loss.
# Questo è più corretto che salvare semplicemente l'ultima epoca.
best_val_loss = float("inf")

best_epoch = -1
best_model_wts = copy.deepcopy(model.state_dict())
patience_counter = 0

start_time = time.time()

for epoch in range(NUM_EPOCHS):
    print(f"\n===== EPOCA {epoch+1}/{NUM_EPOCHS} =====")

    train_loss, train_acc_rf, train_acc_trans = train_epoch(model, train_loader, optimizer, DEVICE)
    val_loss, val_acc_rf, val_acc_trans, _, _, _, _ = validate(model, val_loader, DEVICE)

    history["train_loss"].append(train_loss)
    history["train_acc_rf"].append(train_acc_rf)
    history["train_acc_trans"].append(train_acc_trans)
    history["val_loss"].append(val_loss)
    history["val_acc_rf"].append(val_acc_rf)
    history["val_acc_trans"].append(val_acc_trans)

    print(f"Train Loss: {train_loss:.4f} | Train RF Acc: {train_acc_rf:.4f} | Train Trans Acc: {train_acc_trans:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   RF Acc: {val_acc_rf:.4f} | Val   Trans Acc: {val_acc_trans:.4f}")

    # Se la validation loss migliora abbastanza, salviamo un nuovo checkpoint.
    # Altrimenti aumentiamo il contatore della patience.
    if val_loss < best_val_loss - MIN_DELTA:
        best_val_loss = val_loss
        best_epoch = epoch + 1
        best_model_wts = copy.deepcopy(model.state_dict())
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        patience_counter = 0
        print(f"Nuovo best model salvato in epoca {best_epoch}")
    else:
        patience_counter += 1
        print(f"Nessun miglioramento. Patience: {patience_counter}/{PATIENCE}")

    if patience_counter >= PATIENCE:
        print("Early stopping attivato.")
        break

elapsed_time = time.time() - start_time
print(f"\nTraining completato in {elapsed_time/60:.2f} minuti.")

model.load_state_dict(best_model_wts)
print(f"Caricato il miglior modello ottenuto all'epoca {best_epoch} con val_loss={best_val_loss:.4f}")

save_history_to_csv(history, HISTORY_CSV_PATH)
print(f"History salvata in: {HISTORY_CSV_PATH}")

#**Evaluation/Test**

## **5. Valutazione e Analisi delle Tracce Cross-Classe**

In questa fase finale analizziamo l'accuratezza del modulo di classificazione Real/Fake disaggregando i dati per ciascuna specifica trasformazione di post-processing subita. Questo ci permette di verificare l'esistenza di tracce cross-classe e di identificare quali alterazioni siano statisticamente più distruttive nei confronti delle impronte digitali geometrico-strutturali lasciate dai software di generazione IA.

In [ ]:
# Estrazione dei risultati dettagliati sul validation set
_, final_acc_rf, final_acc_trans, rf_preds, rf_labels, trans_preds, trans_labels = validate(model, val_loader, DEVICE)


# La confusion matrix ci fa vedere non solo quante predizioni sono corrette,
# ma anche quali classi vengono confuse più spesso dal modello.
rf_cm = confusion_matrix(rf_labels, rf_preds)
trans_cm = confusion_matrix(trans_labels, trans_preds)

plot_confusion_matrix(rf_cm, ["Real", "Fake"], title="Confusion Matrix - Real/Fake")
plot_confusion_matrix(trans_cm, list(TRANS_LABELS.keys()), title="Confusion Matrix - Transformation")

rf_preds = np.array(rf_preds)
rf_labels = np.array(rf_labels)
trans_labels = np.array(trans_labels)

print("=======================================================")
print("  REPORT DI CLASSIFICAZIONE: COMPITO BINARIO REAL/FAKE  ")
print("=======================================================")
print(classification_report(rf_labels, rf_preds, target_names=["Real", "Fake"]))

print("\n=======================================================")
print("  REPORT DI CLASSIFICAZIONE: COMPITO POST-PROCESSING   ")
print("=======================================================")
print(classification_report(trans_labels, trans_preds, target_names=list(TRANS_LABELS.keys())))

print("\n=======================================================")
print(" ANALISI DISAGGREGATA DELL'IMPATTO DELLE ALTERAZIONI   ")
print("=======================================================")

for trans_name, trans_idx in TRANS_LABELS.items():
    # Seleziona solo i campioni che hanno subito la specifica trasformazione corrente
    mask = (trans_labels == trans_idx)
    subset_labels = rf_labels[mask]
    subset_preds = rf_preds[mask]

    sub_acc = accuracy_score(subset_labels, subset_preds)
    print(f"\nAccuratezza complessiva Real/Fake su immagini '{trans_name.upper()}': {sub_acc:.4f}")

    # Divisione interna tra Foto Reali e Immagini Fake sottoposte alla medesima alterazione
    real_mask = (subset_labels == 0)
    fake_mask = (subset_labels == 1)

    acc_on_real = accuracy_score(subset_labels[real_mask], subset_preds[real_mask]) if sum(real_mask) > 0 else 0
    acc_on_fake = accuracy_score(subset_labels[fake_mask], subset_preds[fake_mask]) if sum(fake_mask) > 0 else 0
    print(f"  -> Accuratezza specifica su FOTO REALI: {acc_on_real:.4f}")
    print(f"  -> Accuratezza specifica su IMMAGINI IA (FAKE): {acc_on_fake:.4f}")


# Salviamo tutte le metriche principali in JSON per avere un riepilogo ordinato e riutilizzabile.
final_metrics = {
    "best_epoch": best_epoch,
    "best_val_loss": best_val_loss,
    "final_val_acc_rf": final_acc_rf,
    "final_val_acc_trans": final_acc_trans,
    "rf_classification_report": classification_report(
        rf_labels, rf_preds, target_names=["Real", "Fake"], output_dict=True
    ),
    "trans_classification_report": classification_report(
        trans_labels, trans_preds, target_names=list(TRANS_LABELS.keys()), output_dict=True
    )
}

save_metrics_to_json(final_metrics, METRICS_PATH)
print(f"Metriche finali salvate in: {METRICS_PATH}")

In [ ]:
# ──────────────────────────────────────────────────────────────────
# DOWNLOAD DIRETTO SUL PC
# ──────────────────────────────────────────────────────────────────
# Scarica direttamente sul computer i tre file prodotti dal training.
# Eseguire PRIMA di chiudere Colab, altrimenti i file
# vengono persi quando la sessione termina.
#
# I file scaricati sono:
#   - best_multitask_model.pt  → pesi del miglior modello (early stopping)
#   - multitask_metrics.json   → metriche finali di valutazione (accuracy, F1, report)
#   - multitask_history.csv    → loss e accuracy epoca per epoca (train + val)
# ──────────────────────────────────────────────────────────────────

#from google.colab import files

#files.download(str(BEST_MODEL_PATH))
#files.download(str(METRICS_PATH))
#files.download(str(HISTORY_CSV_PATH))

print("Download avviato per i tre file.")